# CaRL × Luna: saved campaign evidence

This notebook runs **offline by default** and reads deterministic CSV exports only. It makes no model calls and does not launch a solver. Current pilot: one real 12×12×4 family; 22 completed runs, 11 complete pairs at B=256, no solves, two interrupted checkpoints and 576 missing cells. The stopped pilot is incomplete. There are no full-solver 32×32 pilot observations.

Run with the checkout's `.venv/bin/python` kernel. Rebuild inputs separately with `uv run --extra icl python -m scripts.export_campaign_plot_data`. The old mock interface notebook is retained as `luna_icl_evaluation_mock_legacy.ipynb`.
If automatic checkout discovery fails, set `REPO_PATH = '/path/to/CaRL'` before running the setup cell (or set the `REPO_PATH` environment variable). Set `DATA_PATH` to inspect another directory of exported CSV tables; coverage is read from those tables.


In [ ]:
import gzip
import json
import os
import sys
from pathlib import Path
from typing import Any, cast

OFFLINE = True
if not OFFLINE:
    raise ValueError('This notebook supports offline evaluation only.')
os.environ['HF_HUB_OFFLINE'] = '1'

def deny_external_connections(event: str, arguments: tuple[Any, ...]) -> None:
    if event == 'socket.connect':
        address = arguments[1]
        if isinstance(address, tuple) and address[0] not in ('127.0.0.1', '::1', 'localhost'):
            raise RuntimeError('External network disabled in offline evaluation notebook')

sys.addaudithook(deny_external_connections)
REPO_PATH = globals().get('REPO_PATH', os.environ.get('REPO_PATH'))
if REPO_PATH is not None:
    candidate = Path(REPO_PATH).expanduser().resolve()
    ROOT = candidate if (candidate / 'scripts/plot_data_export').is_dir() else None
else:
    anchors = (Path.cwd(), *Path.cwd().parents, *Path(sys.executable).parents)
    ROOT = next((p.resolve() for anchor in anchors for p in (anchor, anchor / 'CaRL')
                 if (p / 'scripts/plot_data_export').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Set REPO_PATH to the CaRL checkout path, then rerun this cell.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd  # noqa: E402 -- install offline guard and checkout path first.
import plotly.io as pio  # noqa: E402 -- install offline guard and checkout path first.
from IPython.display import display  # noqa: E402 -- install offline guard and checkout path first.
from plotly.subplots import make_subplots  # noqa: E402 -- install offline guard and checkout path first.
from scripts.render_campaign_plot_data import board_heatmap, make_figures  # noqa: E402 -- install offline guard and checkout path first.

pio.renderers.default = 'plotly_mimetype'
DATA_PATH = globals().get('DATA_PATH')  # Optional directory containing the exported CSV tables.
DATA = Path(DATA_PATH).expanduser().resolve() if DATA_PATH is not None else ROOT / 'analysis/luna_icl/campaign_100/plot_data'
print('OFFLINE:', OFFLINE, '| Inputs:', DATA)


## Coverage and denominators

Empty `solved` fields are missing outcomes, never failures. Completed, replay-verified observations form the denominator; early local-limit stops are excluded. Missing B=128 and B=512 pilot cells have no solve-rate point. Validation probes below belong to a separate split.

In [ ]:
runs = pd.read_csv(DATA / 'runs.csv')
rates = pd.read_csv(DATA / 'solve_rate_by_budget.csv')
pairs = pd.read_csv(DATA / 'paired_deltas.csv')
validation = json.loads((DATA / 'validation.json').read_text())
assert not runs.empty and runs.run_key.is_unique
assert not rates.duplicated(['family', 'method', 'budget_actions']).any()
assert runs.loc[runs.execution_status != 'complete', 'solved'].isna().all()
display(cast(pd.Series, runs.groupby(['budget_actions', 'method', 'execution_status']).size()).rename('runs').reset_index())
display(rates[['family', 'method', 'budget_actions', 'verified_successes', 'denominator_boards',
               'expected_boards', 'solve_rate', 'wilson_low', 'wilson_high']])
display(pd.DataFrame([validation]))
figures = make_figures(DATA)


## A. Independently executed action budgets

Wilson 95% intervals apply to the observed single-seed board outcomes. The paired difference uses common board IDs. With 0/11 for each method, the paired bootstrap collapses to zero; this does not establish equivalence. The unobserved 89 pairs allow the full-cohort difference to range from −89 to +89 percentage points. Missing budgets are not interpolated, and the generated-family panel explicitly marks the absence of observations.

In [ ]:
figures['A_solve_rate'].show()
display(pairs)


## B. Measured training calibration

`k` is a trajectory action offset, `p_hat` an observed exact-hit frequency, and `d` the mean successful first-hit time. Successful target weights define the within-context ECDF; tied times share a jump. The displayed quantile is `min d` such that `F(d) >= f`. Zero-hit targets have no `d`, `w` or `u`. Three training contexts are selected deterministically by the most distinct successful times, then successful target count, then context ID. This is an illustrative selection rule, not a preregistered experiment. No pilot board enters calibration.

In [ ]:
figures['B_calibration'].show()
display(pd.read_csv(DATA / 'figure1_quantiles.csv'))
display(pd.read_csv(DATA / 'reachability_by_offset.csv'))


## C. Component observations

CLLP exact target hits, generator reachability and value estimates are component diagnostics, not full-board solve rates. Query failures remain in the denominator. Value reference trajectories are verified continuations, not necessarily shortest paths. Historical component variants are labelled separately.

In [ ]:
figures['C_components'].show()
policy = pd.read_csv(DATA / 'component_policy.csv')
display(policy.groupby(['variant', 'k']).exact_hit.agg(['sum', 'count']))


## D. Resource use and stop reasons

The figure uses recorded pilot runs and their ledger tags. Settled cost and unknown reservations remain separate. Provider usage excludes local replay; elapsed times with mixed caching do not estimate live latency. The stage table includes historical campaign activity and any separately tagged exploratory work; those rows are not added to the pilot bill. Calibration is a previously incurred shared cost and must be counted once.

In [ ]:
figures['D_cost_stops'].show()
costs = pd.read_csv(DATA / 'component_costs.csv')
display(costs.groupby('stage')[['provider_attempts', 'cost_settled_usd', 'cost_unknown_reserved_usd']].sum())


## A recorded search transition

Choose `TRANSITION_INDEX` and rerun this cell to inspect a saved generator attempt. These are real source, proposed and reached boards from a production trace. A proposal may fail; its endpoint is never replaced by its intended target. Interrupted traces retain their status.

In [ ]:
with gzip.open(DATA / 'search_attempts.jsonl.gz', 'rt', encoding='utf-8') as source:
    transitions = [json.loads(line) for line in source]
proposed = [row for row in transitions if row['proposal'] is not None]
TRANSITION_INDEX = 0
attempt = proposed[TRANSITION_INDEX]
view = make_subplots(rows=1, cols=3, subplot_titles=('Source', 'Proposed goal', 'Actual endpoint'))
for column, field in enumerate(('source', 'proposal', 'actual_endpoint'), start=1):
    view.add_trace(board_heatmap(attempt[field]), row=1, col=column)
    view.update_yaxes(autorange='reversed', row=1, col=column)
view.update_layout(height=450, title=f"{attempt['method']} · scale={attempt['scale']} · reached={attempt['reached']}")
view.show()
display(pd.DataFrame([{key: value for key, value in attempt.items()
                       if key not in ('source', 'proposal', 'actual_endpoint')}]))


## Saved Torch versus Luna: two validation boards at B=128

The pretrained fixed-offset BART/BERT checkpoints solve both recorded boards; the selected Luna relative and absolute configurations solve neither. Board IDs, initial-state hashes, action budgets and seed match. Torch uses greedy argmax; Luna samples its categorical policy. Torch checkpoint training overlap with these boards is unknown. The scale-validation boards were used to choose Luna scales, and this is a two-board engineering comparison, not an unbiased benchmark or a causal test of relative scaling. Both use the shared action-budget search; this is not original node-budget AdaSubS.

In [ ]:
comparison = pd.read_csv(DATA / 'torch_luna_validation_runs.csv')
display(pd.read_csv(DATA / 'torch_luna_validation_summary.csv'))
display(comparison[['board_id', 'method', 'budget_actions', 'solved', 'attempted_actions',
                    'solution_length', 'expansions', 'policy_mode', 'replay_verified']])
assert comparison.groupby('method').board_id.nunique().eq(2).all()
assert comparison.replay_verified.all()


## Reproducible figures and remaining evidence

Editable figures are in `figures`; the standalone renderer exports self-contained HTML plus SVG/PDF:

```bash
uv run --extra icl python -m scripts.render_campaign_plot_data --formats html svg pdf
```

The CSV data dictionary is `plot_data/README.md`; hashes are in `manifest.json`. Current evidence cannot establish a relative-scale advantage, a complete action-budget curve, a full 100-board pilot result, generalization to 32×32, or a 1000-board holdout result. Full 32×32 solves are outside the current scope. The large pilot is stopped and must not be resumed by this notebook.